# 可选实验 - Softmax 函数
在本实验中，我们将探索 softmax 函数。该函数在 Softmax 回归以及用神经网络解决多类别分类问题时都会用到。  

<center>  <img  src="./图片/C2_W2_Softmax_Header.PNG" width="600" />

> **图中文字（中文）**：逻辑回归 Softmax 回归（4 个可能的输出）： a1 = e^z1/(e^z1 + e^z2 + e^z3 + e^z4) = P(y = 1|x) = 0.30 a2 = … = P(y = 2|x) = 0.20 a3 = … = P(y = 3|x) = 0.15 a4 = … = P(y = 4|x) = 0.35 带 Softmax 输出的神经网络 输入 → 25 个单元 → 15 个单元 → 10 个单元 最后一层做逻辑回归，输出经 softmax Softmax 回归（N 个可能的输出）：a_j = P(y = j|x)  <center/>

  

In [1]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('./deeplearning.mplstyle')
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from IPython.display import display, Markdown, Latex
from sklearn.datasets import make_blobs
%matplotlib widget
from matplotlib.widgets import Slider
from lab_utils_common import dlc
from lab_utils_softmax import plt_softmax
import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)
tf.autograph.set_verbosity(0)

> **注意**：通常在本课程的 notebook 中，计数采用从 0 开始到 N-1 结束的约定，即 $\\sum_{i=0}^{N-1}$；而课程视频从 1 开始到 N 结束，即 $\\sum_{i=1}^{N}$。这是因为代码通常从 0 开始迭代，而在课程中从 1 数到 N 会得到更简洁清晰的公式。本 notebook 的公式比一般实验更多，因此将打破这一约定，采用 1 到 N 计数。

## Softmax 函数
在 softmax 回归和带 Softmax 输出的神经网络中，都会生成 N 个输出，并选择其中一个作为预测类别。两种情况下都由一个线性函数生成向量 $\\mathbf{z}$，再把 $\\mathbf{z}$ 送入 softmax 函数。softmax 函数按下面所述把 $\\mathbf{z}$ 转换为概率分布。应用 softmax 之后，每个输出都在 0 和 1 之间，且各输出之和为 1，因此可以解释为概率。输入越大，对应的输出概率越大。
<center>  <img  src="./图片/C2_W2_SoftmaxReg_NN.png" width="600" />

> **图中文字（中文）**：Softmax 回归：a1 = P(y = 1|x) = …，…，a10 = P(y = 10|x) = … Softmax 函数 带 Softmax 输出的神经网络  

softmax 函数可以写作：
$$a_j = \\frac{e^{z_j}}{ \\sum_{k=1}^{N}{e^{z_k} }} \\tag{1}$$
输出 $\\mathbf{a}$ 是长度为 N 的向量，所以对于 softmax 回归，也可以写作：
\\begin{align}
\\mathbf{a}(x) =
\\begin{bmatrix}
P(y = 1 | \\mathbf{x}; \\mathbf{w},b) \\\\
\\vdots \\\\
P(y = N | \\mathbf{x}; \\mathbf{w},b)
\\end{bmatrix}
=
\\frac{1}{ \\sum_{k=1}^{N}{e^{z_k} }}
\\begin{bmatrix}
e^{z_1} \\\\
\\vdots \\\\
e^{z_{N}} \\\\
\\end{bmatrix} \\tag{2}
\\end{align}


这表明输出是一个概率向量。第一个条目是在给定输入 $\\mathbf{x}$ 和参数 $\\mathbf{w}$、$\\mathbf{b}$ 时输入属于第一个类别的概率。  
让我们写一个 NumPy 实现：

In [2]:
def my_softmax(z):
    ez = np.exp(z)              #逐元素指数
    sm = ez/np.sum(ez)
    return(sm)

下面，用滑块改变 `z` 各输入的值。

In [3]:
plt.close("all")
plt_softmax(my_softmax)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

在你改变上面 z 值的过程中，有几点需要注意：
* softmax 分子中的指数会放大小数值之间的差异
* 各输出值之和为 1
* softmax 作用于所有输出。例如，改变 `z0` 会改变 `a0`-`a3` 的值。把它与 ReLU 或 Sigmoid 等其他激活函数比较一下，后者是单输入单输出。

## 代价
<center> <img  src="./图片/C2_W2_SoftMaxCost.png" width="400" />

> **图中文字（中文）**：代价：Softmax 回归 vs 逻辑回归 逻辑回归：loss = −y·log(a1) − (1 − y)·log(1 − a1) Softmax 回归：a_j = e^z_j/(e^z1 + e^z2 + … + e^zN) 交叉熵损失 = −log a2 若 y = 2；…；−log aN 若 y = N J(w,b) = 损失的平均值    <center/>

与 Softmax 相关的损失函数是交叉熵损失：
\\begin{equation}
  L(\\mathbf{a},y)=\\begin{cases}
    -log(a_1), & \\text{if $y=1$}.\\\\
        &\\vdots\\\\
     -log(a_N), & \\text{if $y=N$}
  \\end{cases} \\tag{3}
\\end{equation}

其中 y 是该样本的目标类别，$\\mathbf{a}$ 是 softmax 函数的输出。特别地，$\\mathbf{a}$ 中的取值是概率，且之和为 1。
>**回忆一下：** 在本课程中，Loss 针对单个样本，而 Cost 涵盖所有样本。
 
 
注意上面 (3) 式中，只有与目标对应的那一行对损失有贡献，其他行都为零。要写出代价方程，我们需要一个“指示函数”，当索引与目标匹配时取 1，否则取 0。
    $$\\mathbf{1}\\{y == n\\} = =\\begin{cases}
    1, & \\text{if $y==n$}.\\\\
    0, & \\text{otherwise}.
  \\end{cases}$$
于是代价为：
\\begin{align}
J(\\mathbf{w},b) = - \\left[ \\sum_{i=1}^{m} \\sum_{j=1}^{N}  1\\left\\{y^{(i)} == j\\right\\} \\log \\frac{e^{z^{(i)}_j}}{\\sum_{k=1}^N e^{z^{(i)}_k} }\\right] \\tag{4}
\\end{align}

其中 $m$ 是样本数，$N$ 是输出数。这是所有损失的平均值。


## Tensorflow
本实验将讨论在 Tensorflow 中实现 softmax 交叉熵损失的两种方式：“显而易见”的方式和“首选”的方式。前者最直接，后者数值上更稳定。

让我们先创建一个数据集来训练多类别分类模型。

In [4]:
# 为示例构造数据集
centers = [[-5, 2], [-2, -2], [1, 2], [5, -2]]
X_train, y_train = make_blobs(n_samples=2000, centers=centers, cluster_std=1.0,random_state=30)

### *显而易见*的组织方式

下面的模型把 softmax 作为最后一个 Dense 层的激活函数来实现。
损失函数在 `compile` 指令中单独指定。

损失函数是 `SparseCategoricalCrossentropy`，即上面 (3) 式描述的损失。在这个模型中，softmax 发生在最后一层。损失函数接收 softmax 的输出，即一个概率向量。


In [5]:
model = Sequential(
    [ 
        Dense(25, activation = 'relu'),
        Dense(15, activation = 'relu'),
        Dense(4, activation = 'softmax')    # < 这里使用 softmax 激活函数
    ]
)
model.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    optimizer=tf.keras.optimizers.Adam(0.001),
)

model.fit(
    X_train,y_train,
    epochs=10
)
        

Epoch 1/10
63/63 [==============================] - 0s 958us/step - loss: 0.8390
Epoch 2/10
63/63 [==============================] - 0s 1ms/step - loss: 0.3807
Epoch 3/10
63/63 [==============================] - 0s 1ms/step - loss: 0.1923
Epoch 4/10
63/63 [==============================] - 0s 1ms/step - loss: 0.1094
Epoch 5/10
63/63 [==============================] - 0s 1ms/step - loss: 0.0751
Epoch 6/10
63/63 [==============================] - 0s 1ms/step - loss: 0.0602
Epoch 7/10
63/63 [==============================] - 0s 950us/step - loss: 0.0510
Epoch 8/10
63/63 [==============================] - 0s 1ms/step - loss: 0.0449
Epoch 9/10
63/63 [==============================] - 0s 1ms/step - loss: 0.0408
Epoch 10/10
63/63 [==============================] - 0s 1ms/step - loss: 0.0377


由于 softmax 被集成到输出层，输出是一个概率向量。

In [6]:
p_nonpreferred = model.predict(X_train)
print(p_nonpreferred [:2])
print("largest value", np.max(p_nonpreferred), "smallest value", np.min(p_nonpreferred))

[[1.66e-03 1.52e-03 9.85e-01 1.16e-02]
 [9.93e-01 6.35e-03 4.36e-04 8.06e-06]]
largest value 0.99999964 smallest value 2.3815372e-11


### 首选方式 <img align="Right" src="./图片/C2_W2_softmax_accurate.png"  style=" width:400px; padding: 10px 20px ; ">

> **图中文字（中文）**：Softmax 的数值上更精确的实现 Softmax 回归：loss = −log a1 若 y = 1；…；loss = −log a10 若 y = 10 loss = −log(e^z1 + … + e^z10) 若 y = 10（更精确） model.compile(loss=SparseCategoricalCrossentropy(from_logits=True))
回忆一下课上所讲：如果在训练时把 softmax 和损失合并在一起，可以得到更稳定、更准确的结果。这里展示的“首选”组织方式就能实现这一点。


在首选方式中，最后一层使用线性激活。出于历史原因，这种形式的输出被称为 *logits*。损失函数多了一个参数：`from_logits = True`。它告诉损失函数，softmax 运算应包含在损失计算中。这样就能采用优化过的实现。

In [7]:
preferred_model = Sequential(
    [ 
        Dense(25, activation = 'relu'),
        Dense(15, activation = 'relu'),
        Dense(4, activation = 'linear')   #<-- 注意
    ]
)
preferred_model.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),  #<-- 注意
    optimizer=tf.keras.optimizers.Adam(0.001),
)

preferred_model.fit(
    X_train,y_train,
    epochs=10
)
        

Epoch 1/10
63/63 [==============================] - 0s 949us/step - loss: 1.0512
Epoch 2/10
63/63 [==============================] - 0s 937us/step - loss: 0.3729
Epoch 3/10
63/63 [==============================] - 0s 1ms/step - loss: 0.1545
Epoch 4/10
63/63 [==============================] - 0s 1ms/step - loss: 0.0956
Epoch 5/10
63/63 [==============================] - 0s 923us/step - loss: 0.0725
Epoch 6/10
63/63 [==============================] - 0s 1ms/step - loss: 0.0607
Epoch 7/10
63/63 [==============================] - 0s 1ms/step - loss: 0.0531
Epoch 8/10
63/63 [==============================] - 0s 1ms/step - loss: 0.0475
Epoch 9/10
63/63 [==============================] - 0s 1ms/step - loss: 0.0431
Epoch 10/10
63/63 [==============================] - 0s 1ms/step - loss: 0.0396


#### 输出的处理
注意，在首选模型中，输出不是概率，其取值范围可以从很大的负数到很大的正数。当需要做期望得到概率的预测时，必须把输出送入 softmax。
让我们看看首选模型的输出：

In [8]:
p_preferred = preferred_model.predict(X_train)
print(f"two example output vectors:\n {p_preferred[:2]}")
print("largest value", np.max(p_preferred), "smallest value", np.min(p_preferred))

two example output vectors:
 [[-2.94 -2.33  2.86 -1.25]
 [ 1.5  -4.28 -7.08 -7.93]]
largest value 8.857447 smallest value -13.404879


输出的预测值不是概率！
如果希望得到概率，输出应经过 [softmax](https://www.tensorflow.org/api_docs/python/tf/nn/softmax) 处理。

In [9]:
sm_preferred = tf.nn.softmax(p_preferred).numpy()
print(f"two example output vectors:\n {sm_preferred[:2]}")
print("largest value", np.max(sm_preferred), "smallest value", np.min(sm_preferred))

two example output vectors:
 [[2.97e-03 5.46e-03 9.75e-01 1.62e-02]
 [9.97e-01 3.08e-03 1.86e-04 8.00e-05]]
largest value 0.99999774 smallest value 1.0387312e-07


要选出最可能的类别，并不需要 softmax。可以用 [np.argmax()](https://numpy.org/doc/stable/reference/generated/numpy.argmax.html) 找到最大输出的索引。

In [10]:
for i in range(5):
    print( f"{p_preferred[i]}, category: {np.argmax(p_preferred[i])}")

[-2.94 -2.33  2.86 -1.25], category: 2
[ 1.5  -4.28 -7.08 -7.93], category: 0
[ 1.02 -2.93 -5.43 -6.26], category: 0
[-2.19  3.48 -1.81 -2.91], category: 1
[-2.32 -6.31  3.67 -4.91], category: 2


## SparseCategorialCrossentropy 与 CategoricalCrossEntropy
Tensorflow 的目标值有两种可能的格式，选择哪种损失函数就决定了期望哪一种。
- SparseCategorialCrossentropy：期望目标是与索引对应的整数。例如，如果有 10 个可能的目标值，y 将在 0 到 9 之间。
- CategoricalCrossEntropy：期望样本的目标值采用独热编码，即目标索引处为 1，其余 N-1 个条目为 0。对于有 10 个可能目标值、目标是 2 的样本，应为 [0,0,1,0,0,0,0,0,0,0]。


## 恭喜！
在本实验中你
- 更熟悉了 softmax 函数，以及它在 softmax 回归和神经网络 softmax 激活中的应用。
- 学会了 Tensorflow 中的首选模型构建方式：
    - 最后一层不加激活函数（等同于线性激活）
    - SparseCategoricalCrossentropy 损失函数
    - 使用 from_logits=True
- 认识到与 ReLU 和 Sigmoid 不同，softmax 作用于多个输出。

## 数值稳定性（可选）
本节讨论一些用于提高数值稳定性的方法。这部分是给感兴趣的读者看的，完全不是必需内容。

### Softmax 的数值稳定性
softmax 的输入是线性层的输出 $z_j = \\mathbf{w_j} \\cdot \\mathbf{x}^{(i)}+b$。这些可能是很大的数。softmax 算法的第一步是计算 $e^{z_j}$。如果数值太大，可能导致溢出错误。试着运行下面的单元格：

In [11]:
for z in [500,600,700,800]:
    ez = np.exp(z)
    zs = "{" + f"{z}" + "}"
    print(f"e^{zs} = {ez:0.2e}")

e^{500} = 1.40e+217
e^{600} = 3.77e+260
e^{700} = 1.01e+304
e^{800} = inf


如果指数太大，该运算会产生溢出。自然地，`my_softmax()` 也会产生同样的错误：

In [12]:
z_tmp = np.array([[500,600,700,800]])
my_softmax(z_tmp)

array([[ 0.,  0.,  0., nan]])

可以通过减小指数的大小来提高数值稳定性。
回忆一下
$$ e^{a + b} = e^ae^b$$
如果 $b$ 与 $a$ 符号相反，就会减小指数的大小。具体来说，如果你给 softmax 乘上一个分数：
$$a_j = \\frac{e^{z_j}}{ \\sum_{i=1}^{N}{e^{z_i} }} \\frac{e^{-b}}{ {e^{-b}}}$$
指数会被减小，而 softmax 的值不变。如果 $e^b$ 中的 $b$ 取 $z_j$ 的最大值 $max_j(\\mathbf{z})$，指数就会被减小到最小值。
$$\\begin{align}
a_j &= \\frac{e^{z_j}}{ \\sum_{i=1}^{N}{e^{z_i} }} \\frac{e^{-max_j(\\mathbf{z})}}{ {e^{-max_j(\\mathbf{z})}}} \\\\
&= \\frac{e^{z_j-max_j(\\mathbf{z})}}{ \\sum_{i=1}^{N}{e^{z_i-max_j(\\mathbf{z})} }} 
\\end{align}$$
习惯上记 $C=max_j(\\mathbf{z})$，因为对任意常数 C 该等式都成立。


$$
a_j = \\frac{e^{z_j-C}}{ \\sum_{i=1}^{N}{e^{z_i-C} }} \\quad\\quad\\text{where}\\quad C=max_j(\\mathbf{z})\\tag{5}
$$

如果我们看看那个麻烦的例子，其中 $\\mathbf{z}$ 包含 500,600,700,800，则 $C=max_j(\\mathbf{z})=800$：
\\begin{align}
\\mathbf{a}(x) =
\\frac{1}{ e^{500-800} + e^{600-800} + e^{700-800} + e^{800-800}}
\\begin{bmatrix}
e^{500-800} \\\\
e^{600-800} \\\\
e^{700-800} \\\\
e^{800-800} \\\\
\\end{bmatrix}
= 
\\begin{bmatrix}
5.15e-131 \\\\
1.38e-87 \\\\
3.7e-44 \\\\
1.0 \\\\
\\end{bmatrix}
\\end{align}

让我们重写 `my_softmax` 以提高其数值稳定性。

In [18]:
def my_softmax_ns(z):
    """numerically stablility improved"""
    bigz = np.max(z)
    ez = np.exp(z-bigz)              # 让指数最小化
    sm = ez/np.sum(ez)
    return(sm)

让我们试一下，并与 tensorflow 的实现做比较：

In [20]:
z_tmp = np.array([500.,600,700,800])
print(tf.nn.softmax(z_tmp).numpy(), "\n", my_softmax_ns(z_tmp))

[5.15e-131 1.38e-087 3.72e-044 1.00e+000] 
 [5.15e-131 1.38e-087 3.72e-044 1.00e+000]


大数值不再引起溢出了。

### 交叉熵损失的数值稳定性

与 Softmax 相关的损失函数即交叉熵损失，这里再写一遍：
\\begin{equation}
  L(\\mathbf{a},y)=\\begin{cases}
    -log(a_1), & \\text{if $y=1$}.\\\\
        &\\vdots\\\\
     -log(a_N), & \\text{if $y=N$}
  \\end{cases} 
\\end{equation}

其中 y 是该样本的目标类别，$\\mathbf{a}$ 是 softmax 函数的输出。特别地，$\\mathbf{a}$ 中的取值是概率，且之和为 1。
让我们考虑目标为二（$y=2$）的情形，只看该情形下的损失。此时损失为：  
$$L(\\mathbf{a})= -log(a_2)$$
回忆一下，$a_2$ 是上面所述 softmax 函数的输出，因此可以写作：
$$L(\\mathbf{z})= -log\\left(\\frac{e^{z_2}}{ \\sum_{i=1}^{N}{e^{z_i} }}\\right) \\tag{6}$$
这可以进一步优化。不过要做这些优化，softmax 和损失必须一起计算，正如你上面看到的 Tensorflow“首选”实现那样。

从上面的 (6) 式出发，y=2 情形下的损失：
由 $log(\\frac{a}{b}) = log(a) - log(b)$，(6) 式可改写为：
$$L(\\mathbf{z})= -\\left[log(e^{z_2}) - log \\sum_{i=1}^{N}{e^{z_i} }\\right] \\tag{7}$$
第一项可以简化为 $z_2$：
$$L(\\mathbf{z})= -\\left[z_2 - log( \\sum_{i=1}^{N}{e^{z_i} })\\right] =  \\underbrace{log \\sum_{i=1}^{N}{e^{z_i} }}_\\text{logsumexp()} -z_2 \\tag{8}$$
结果表明，上式中的 $log \\sum_{i=1}^{N}{e^{z_i} }$ 项使用得如此频繁，许多库都有其实现。在 Tensorflow 中就是 tf.math.reduce_logsumexp()。这个求和的一个问题是，如果 $z_i$ 很大，其中的指数可能溢出。为解决这一点，我们可能想和上面一样减去 $e^{max_j(\\mathbf{z})}$，但这需要一点推导：
$$
\\begin{align}
   log \\sum_{i=1}^{N}{e^{z_i} } &= log \\sum_{i=1}^{N}{e^{(z_i - max_j(\\mathbf{z}) + max_j(\\mathbf{z}))}} \\tag{9}\\\\
                          &= log \\sum_{i=1}^{N}{e^{(z_i - max_j(\\mathbf{z}))} e^{max_j(\\mathbf{z})}} \\\\
                          &= log(e^{max_j(\\mathbf{z})}) + log \\sum_{i=1}^{N}{e^{(z_i - max_j(\\mathbf{z}))}} \\\\
                          &= max_j(\\mathbf{z})  + log \\sum_{i=1}^{N}{e^{(z_i - max_j(\\mathbf{z}))}}
\\end{align}
$$
现在指数就不太可能溢出了。习惯上记 $C=max_j(\\mathbf{z})$，因为对任意常数 C 该等式都成立。于是我们可以写出损失方程：
    
$$L(\\mathbf{z})= C+ log( \\sum_{i=1}^{N}{e^{z_i-C} }) -z_2  \\;\\;\\;\\text{where } C=max_j(\\mathbf{z}) \\tag{10} $$
这是计算上更简单、更稳定的损失版本。上式针对目标 y=2 的情形，但可推广到任意目标。